# 04 — Neural Networks: Fraud Detection
**Dataset:** Credit Card Fraud Detection — të dhëna të parapërpunuara nga `02_preprocessing.ipynb`

Qëllimi: Projektojmë dhe trajnojmë dy arkitektura rrjetash neurale për detektim mashtrimi:
1. **Arkitektura 1** — MLP i thjeshtë (Simple MLP): 2 shtresa të fshehta, pa regularizim
2. **Arkitektura 2** — MLP i thellë (Deep MLP): BatchNormalization + Dropout, trajnim me class weights

Të dy modelet krahasohen me metrikat: Accuracy, Precision, Recall, F1-score, ROC-AUC.  
Modeli më i mirë ruhet si `models/best_model.h5`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f'TensorFlow version : {tf.__version__}')
print(f'Keras version      : {keras.__version__}')
print('Bibliotekat u ngarkuan me sukses!')

In [ ]:
X_train       = np.load('../data/processed/X_train.npy')
X_test        = np.load('../data/processed/X_test.npy')
y_train       = np.load('../data/processed/y_train.npy')
y_test        = np.load('../data/processed/y_test.npy')
feature_names = np.load('../data/processed/feature_names.npy', allow_pickle=True)

INPUT_DIM = X_train.shape[1]

print('=' * 55)
print('TË DHËNAT E NGARKUARA')
print('=' * 55)
print(f'X_train : {X_train.shape}  (SMOTE-balanced)')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  | Fraud: {(y_train==1).sum():,}  Legjitime: {(y_train==0).sum():,}')
print(f'y_test  : {y_test.shape}   | Fraud: {(y_test==1).sum():,}   Legjitime: {(y_test==0).sum():,}')
print(f'\nInput dim : {INPUT_DIM} features')
print(f'Features  : {list(feature_names)}')

## 1. Arkitektura 1 — Simple MLP

**Struktura:**
```
Input(30) → Dense(64, ReLU) → Dense(32, ReLU) → Dense(1, Sigmoid)
```

**Zgjedhjet e dizajnit:**
- **ReLU** — aktivizim standard për shtresa të fshehta, shmang problemin e vanishing gradient
- **Sigmoid** — prodhon probabilitet [0,1] për klasifikim binar
- **Binary Crossentropy** — humbja standarde për probleme binare
- **Adam (lr=0.001)** — optimizer adaptiv, performon mirë pa tuning manual
- **EarlyStopping** — ndalon trajnimin kur `val_loss` nuk përmirësohet (patience=10)
- **class_weight** — kompenzon pabalansin e mbetur në training set pas SMOTE

In [ ]:
def build_simple_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1,  activation='sigmoid')
    ], name='Simple_MLP')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

model1 = build_simple_mlp(INPUT_DIM)
model1.summary()

In [ ]:
cw_values = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: cw_values[0], 1: cw_values[1]}
print(f'Class weights: {class_weight_dict}')

callbacks1 = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

history1 = model1.fit(
    X_train, y_train,
    epochs=100,
    batch_size=512,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=callbacks1,
    verbose=1
)

print(f'\nTrajnimi u ndal në epoch: {len(history1.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['loss'],     color='steelblue', linewidth=2, label='Train Loss')
axes[0].plot(history1.history['val_loss'], color='tomato',    linewidth=2, label='Val Loss', linestyle='--')
axes[0].set_title('Arkitektura 1 — Humbja gjatë Trajnimit', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()

axes[1].plot(history1.history['auc'],     color='steelblue', linewidth=2, label='Train AUC')
axes[1].plot(history1.history['val_auc'], color='tomato',    linewidth=2, label='Val AUC', linestyle='--')
axes[1].set_title('Arkitektura 1 — AUC gjatë Trajnimit', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()

plt.suptitle('Simple MLP — Historiku i Trajnimit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/nn1_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Arkitektura 2 — Deep MLP me Regularizim

**Struktura:**
```
Input(30)
  → Dense(128, ReLU) → BatchNormalization → Dropout(0.3)
  → Dense(64,  ReLU) → BatchNormalization → Dropout(0.3)
  → Dense(32,  ReLU)
  → Dense(1,   Sigmoid)
```

**Ndryshimet kryesore ndaj Arkitekturës 1:**
- **Shtresa më të thella (128→64→32)** — kapacitet më i madh për lidhje jolineare
- **BatchNormalization** — normalizon aktivizimet pas çdo shtrese, stabilizon trajnimin dhe lejon learning rate më të lartë
- **Dropout(0.3)** — çaktivizon 30% neurone rastësisht gjatë trajnimit, parandalon overfitting
- **lr=0.0005** — learning rate më i vogël — ndërvepron mirë me BatchNorm
- **patience=15** — i japim model-it kohë shtesë pasi BatchNorm ka konvergjencë më të ngadaltë

In [ ]:
def build_deep_mlp(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation='relu'),

        layers.Dense(1, activation='sigmoid')
    ], name='Deep_MLP')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=[
            keras.metrics.AUC(name='auc'),
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall')
        ]
    )
    return model

model2 = build_deep_mlp(INPUT_DIM)
model2.summary()

In [ ]:
callbacks2 = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-7, verbose=1)
]

history2 = model2.fit(
    X_train, y_train,
    epochs=100,
    batch_size=512,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=callbacks2,
    verbose=1
)

print(f'\nTrajnimi u ndal në epoch: {len(history2.history["loss"])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history2.history['loss'],     color='seagreen', linewidth=2, label='Train Loss')
axes[0].plot(history2.history['val_loss'], color='tomato',   linewidth=2, label='Val Loss', linestyle='--')
axes[0].set_title('Arkitektura 2 — Humbja gjatë Trajnimit', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend()

axes[1].plot(history2.history['auc'],     color='seagreen', linewidth=2, label='Train AUC')
axes[1].plot(history2.history['val_auc'], color='tomato',   linewidth=2, label='Val AUC', linestyle='--')
axes[1].set_title('Arkitektura 2 — AUC gjatë Trajnimit', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()

plt.suptitle('Deep MLP (BatchNorm + Dropout) — Historiku i Trajnimit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/nn2_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history1.history['loss'],     color='steelblue', linewidth=2, label='Ark.1 Train')
axes[0].plot(history1.history['val_loss'], color='steelblue', linewidth=2, label='Ark.1 Val',   linestyle='--')
axes[0].plot(history2.history['loss'],     color='seagreen',  linewidth=2, label='Ark.2 Train')
axes[0].plot(history2.history['val_loss'], color='seagreen',  linewidth=2, label='Ark.2 Val',   linestyle='--')
axes[0].set_title('Krahasimi i Humbjes — Ark.1 vs Ark.2', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Crossentropy')
axes[0].legend(fontsize=9)

axes[1].plot(history1.history['val_auc'], color='steelblue', linewidth=2, label='Ark.1 Val AUC')
axes[1].plot(history2.history['val_auc'], color='seagreen',  linewidth=2, label='Ark.2 Val AUC')
axes[1].set_title('Krahasimi i Val AUC — Ark.1 vs Ark.2', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend(fontsize=9)

plt.suptitle('Arkitektura 1 (Simple MLP) vs Arkitektura 2 (Deep MLP) — Trajnimi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/nn_training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()